# Can room occupancy be predicted from environmental sensors?

This notebook evaluates whether the binary `Occupancy` target can be predicted from temperature, humidity, light, CO₂, and humidity ratio. Because the observations are approximately one minute apart, the primary evaluation uses the **last 20% of observations as a chronological holdout**. This is more realistic than randomly mixing near-duplicate neighboring readings between train and test sets.

**Success criteria:** outperform a majority-class baseline on balanced accuracy, F1, ROC AUC, and average precision; inspect the confusion matrix; and test how dependent performance is on individual sensors.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", 20)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load and validate the data

In [ ]:
DATA_PATH = Path("data.csv")
assert DATA_PATH.exists(), f"Expected {DATA_PATH.resolve()}"

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
expected = {
    "date", "Temperature", "Humidity", "Light", "CO2",
    "HumidityRatio", "Occupancy"
}
assert expected.issubset(df.columns), f"Missing columns: {expected - set(df.columns)}"
assert set(df["Occupancy"].dropna().unique()) <= {0, 1}, "Target must be binary"

df = df.sort_values("date").reset_index(drop=True)
print(f"Rows: {len(df):,} | columns: {df.shape[1]}")
print(f"Period: {df['date'].min()} to {df['date'].max()}")
print(f"Duplicate rows: {df.duplicated().sum():,}")
display(df.head())

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
})
display(quality)

target_counts = df["Occupancy"].value_counts().sort_index()
display(pd.DataFrame({
    "count": target_counts,
    "share": target_counts / len(df),
}).rename_axis("Occupancy"))

There are no missing values. The positive class is the minority, so plain accuracy is not sufficient: a model that always predicts “unoccupied” already gets roughly 77% accuracy. F1, balanced accuracy, and average precision are therefore central metrics.

## 2. Exploratory analysis

In [ ]:
sensor_cols = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]
display(df[sensor_cols + ["Occupancy"]].describe().T)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, sensor_cols):
    for occupied, color in [(0, "#4C78A8"), (1, "#E45756")]:
        values = df.loc[df["Occupancy"] == occupied, col]
        ax.hist(values, bins=35, alpha=0.55, density=True,
                label=f"Occupancy={occupied}", color=color)
    ax.set_title(col)
    ax.set_ylabel("Density")
axes.flat[-1].axis("off")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower right", bbox_to_anchor=(0.88, 0.08))
fig.suptitle("Sensor distributions by occupancy class", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
sample = df.iloc[::30]  # every ~30 minutes, for a readable overview
fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.plot(sample["date"], sample["Light"], color="#F2B134", label="Light")
ax1.set_ylabel("Light")
ax1.set_xlabel("Time")
ax2 = ax1.twinx()
ax2.fill_between(sample["date"], 0, sample["Occupancy"],
                 color="#E45756", alpha=0.25, label="Occupied")
ax2.set_ylabel("Occupancy")
ax2.set_ylim(0, 1.1)
ax1.set_title("Light and occupancy over time (30-minute sampling)")
fig.tight_layout()
plt.show()

## 3. Leakage-aware train/test split

The timestamp itself is excluded from the predictors. It is used only to order observations. The first 80% trains the models and the final 20% is untouched test data. This tests forward generalization and prevents adjacent readings from being randomly split across train and test.

This dataset covers only about 16 days and has two gaps longer than five minutes, so the result is evidence for this room and period—not proof of performance across seasons, buildings, sensor replacements, or changed lighting policy.

In [ ]:
split_idx = int(len(df) * 0.80)
train = df.iloc[:split_idx].copy()
test = df.iloc[split_idx:].copy()

X_train, y_train = train[sensor_cols], train["Occupancy"]
X_test, y_test = test[sensor_cols], test["Occupancy"]

split_summary = pd.DataFrame({
    "rows": [len(train), len(test)],
    "start": [train["date"].min(), test["date"].min()],
    "end": [train["date"].max(), test["date"].max()],
    "occupied_share": [y_train.mean(), y_test.mean()],
}, index=["train", "test"])
display(split_summary)
print("Gaps longer than 5 minutes:", (df["date"].diff() > pd.Timedelta(minutes=5)).sum())

## 4. Compare baseline and predictive models

In [ ]:
models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2_000, class_weight="balanced", random_state=RANDOM_STATE),
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "Histogram gradient boosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.06,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    ),
}


def evaluate(name, model, X_tr=X_train, y_tr=y_train, X_te=X_test, y_te=y_test):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    prob = model.predict_proba(X_te)[:, 1]
    return model, {
        "model": name,
        "accuracy": accuracy_score(y_te, pred),
        "balanced_accuracy": balanced_accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
        "f1": f1_score(y_te, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_te, prob),
        "average_precision": average_precision_score(y_te, prob),
    }


fitted = {}
rows = []
for name, model in models.items():
    fitted[name], result = evaluate(name, model)
    rows.append(result)

results = pd.DataFrame(rows).set_index("model").sort_values("f1", ascending=False)
display(results.style.format("{:.3f}").background_gradient(cmap="Blues", axis=0))

In [ ]:
best_name = results.index[0]
best_model = fitted[best_name]
best_pred = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, display_labels=["Unoccupied", "Occupied"],
    cmap="Blues", colorbar=False, ax=ax
)
ax.set_title(f"Chronological holdout: {best_name}")
plt.show()

errors = test.loc[y_test.to_numpy() != best_pred,
                  ["date", *sensor_cols, "Occupancy"]].copy()
errors["prediction"] = best_pred[y_test.to_numpy() != best_pred]
print(f"Misclassified observations: {len(errors):,} of {len(test):,}")
display(errors.head(10))

## 5. What drives the prediction?

In [ ]:
rf = fitted["Random forest"]
perm = permutation_importance(
    rf, X_test, y_test,
    scoring="balanced_accuracy",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
importance = pd.DataFrame({
    "mean_drop_in_balanced_accuracy": perm.importances_mean,
    "std": perm.importances_std,
}, index=sensor_cols).sort_values("mean_drop_in_balanced_accuracy", ascending=True)

ax = importance["mean_drop_in_balanced_accuracy"].plot.barh(
    xerr=importance["std"], figsize=(8, 4), color="#4C78A8", legend=False
)
ax.set_xlabel("Decrease after shuffling feature")
ax.set_title("Permutation importance on chronological test data")
plt.tight_layout()
plt.show()
display(importance.sort_values("mean_drop_in_balanced_accuracy", ascending=False))

Permutation importance measures the performance loss when a sensor is shuffled. Correlated features can share or mask importance, so this is an operational diagnostic rather than a causal claim.

Next, an ablation retrains the same random forest after removing sensors. This directly asks how well the model transfers into the later time window if a key sensor is unavailable.

In [ ]:
ablations = {
    "All sensors": sensor_cols,
    "Without Light": [c for c in sensor_cols if c != "Light"],
    "Without Light or CO2": [c for c in sensor_cols if c not in {"Light", "CO2"}],
}

ablation_rows = []
for label, cols in ablations.items():
    model = clone(models["Random forest"])
    _, row = evaluate(
        label, model,
        X_train[cols], y_train,
        X_test[cols], y_test,
    )
    ablation_rows.append(row)

ablation_results = pd.DataFrame(ablation_rows).set_index("model")
display(ablation_results.style.format("{:.3f}").background_gradient(cmap="Blues", axis=0))

## Conclusion

**Yes—occupancy is highly predictable in this dataset.** On the final 20% chronological holdout, the best models achieve approximately **0.995 accuracy and 0.991 F1**, with only about 20 errors among 4,112 observations. This is far above the majority baseline, which has zero recall and zero F1 for occupied rooms.

The main caveat is equally important: performance depends heavily on the `Light` sensor. Removing it causes a large drop in the ablation test. The model may partly be learning a room-specific rule such as “lights on means occupied,” which can fail under daylight, altered lighting schedules, sensor drift, or occupants leaving lights on.

**Recommendation:** prediction is feasible for a prototype using all five sensors. Before deployment, validate on a genuinely independent period (ideally other seasons and operating patterns), monitor sensor distributions and class-specific recall, and test failure scenarios for the light sensor. If the application acts in real time, also define whether sensor readings at the prediction instant are available without delay.